# Qwen2.5-VL quick run
Load the dataset from the data folder and run one example through Qwen2.5-VL.

In [ ]:
from pathlib import Path
import sys


cwd = Path.cwd().resolve()
search_paths = [cwd] + list(cwd.parents)
repo_root = next((p for p in search_paths if (p / 'src').exists()), None)
if repo_root is None:
    raise RuntimeError('Could not find repo root with src/ directory')

sys.path.insert(0, str(repo_root))
print(f'Repo root: {repo_root}')
print(sys.executable)

In [ ]:
from IPython.display import display
from src.dataset import load_intent_examples, load_image

# Load a single example
data_dir = repo_root / "data"
jsonl_path = data_dir / "val_annotated.jsonl"
images_dir = data_dir / "images" / "images"

examples = load_intent_examples(
    jsonl_path=str(jsonl_path),
    images_dir=str(images_dir),
    limit=1,
 )
example = examples[0]

print(f"ID: {example['id']}")
print(f"Ambiguous question: {example['ambiguous_question']}")
print(f"Gold clarification: {example['gold_clarification']}")
print(f"Image path: {example['image_path']}")
display(load_image(example['image_path']))

In [ ]:
# Example of running the model and condition on a single example
import json
from src.model import VLMWrapper
from src.conditions.standard import StandardCondition

model = VLMWrapper(model_name = "Qwen/Qwen2.5-VL-3B-Instruct")
condition = StandardCondition(model)
result = condition.run(example)

print(json.dumps(result, indent=2))

In [11]:
import random
import json
from IPython.display import display
from src.dataset import load_intent_examples, load_image
from src.conditions.standard import StandardCondition
from src.conditions.cot import CoTCondition
from src.conditions.at import ATCondition
from src.conditions.at_cot import ATCoTCondition
from src.conditions.at_subtype import ATSubtypeCondition
from src.model import parse_json_output

# Load the full intent split and sample some examples, filtered on the ambiguity category = intent 
data_dir = repo_root / "data"
jsonl_path = data_dir / "val_annotated.jsonl"
images_dir = data_dir / "images" / "images"

all_examples = load_intent_examples(
    jsonl_path=str(jsonl_path),
    images_dir=str(images_dir),
    limit=None,
 )
num_examples = 1
rng = random.Random(42)
sampled = all_examples if len(all_examples) <= num_examples else rng.sample(all_examples, num_examples)

# Define the conditions to run 
conditions = [
    StandardCondition(model),
    #CoTCondition(model),
    #ATCondition(model),
    #ATCoTCondition(model),
    #ATSubtypeCondition(model),
 ]

num_candidates = 2
sample_temperature = 0.7
sample_top_p = 0.9

# Score for CQ 
def score_clarification(text: str) -> int:
    return 0  # TODO: implement a better scoring function for candidate selection

# Run multiple candidates and select the best one according to the scoring function
def run_candidates(cond, ex, n=num_candidates):
    prompt = cond.build_prompt(ex["ambiguous_question"])
    candidates = []
    for _ in range(n):
        raw_output = model.generate(
            ex["image_path"],
            prompt,
            do_sample=True,
            temperature=sample_temperature,
            top_p=sample_top_p,
        )
        parsed = parse_json_output(raw_output)
        question = parsed.get("clarification_question", "").strip()
        candidates.append(
            {
                "question": question,
                "score": score_clarification(question),
                "raw_output": raw_output,
                "parse_failed": parsed.get("_parse_failed", False),
                "reasoning": parsed.get("reasoning", None)
            }
        )
    best = max(candidates, key=lambda c: c["score"])
    return best, candidates

# For each sampled example, run all conditions and print the results
for idx, ex in enumerate(sampled, 1):
    print("=" * 80)
    print(f"Example {idx} | ID: {ex['id']}")
    print(f"Ambiguous question: {ex['ambiguous_question']}")
    print(f"Gold clarification: {ex['gold_clarification']}")
    print(f"Image path: {ex['image_path']}")
    display(load_image(ex["image_path"]))

    for cond in conditions:
        best, candidates = run_candidates(cond, ex)
        result = {
            "id": ex["id"],
            "condition": cond.name,
            "generated_clarification": best["question"],
            "best_score": best["score"],
            "candidates": [
                {"question": c["question"], "score": c["score"]} for c in candidates
            ],
            "gold_clarification": ex["gold_clarification"],
        }
        print(f"\n[{cond.name}]")
        print(json.dumps(result, indent=2))


[standard]
{
  "id": "val_000297",
  "condition": "standard",
  "generated_clarification": "What specific food items can she eat from the table?",
  "best_score": 0,
  "candidates": [
    {
      "question": "What specific food items can she eat from the table?",
      "score": 0
    },
    {
      "question": "What is the food item she is eating in the picture?",
      "score": 0
    }
  ],
  "gold_clarification": "Are you asking which food the girl should eat for more protein?"
}
